# Feature screening: which features look promising?

What is tested, in three cheap steps (train 2015-2021, val 2022, no early
stopping, no tuning):

1. **Univariate signal**: per feature and per generator, the AUC of the
    feature value alone as a score (direction-corrected:
    max(auc, 1-auc)). No model involved.
2. **Feature-group ablation**: per-generator XGBoost with *default*
    values (`xgb.XGBClassifier(tree_method='hist')`: 100 trees, depth 6,
    lr 0.3), fitted once with all 193 features and once per group dropped
    (calendar, price, reservoir state, inflow, constraints, local). The
    drop in micro AUC measures how much each group carries.
3. **Top-10 features only**: the same default XGBoost on only each
    generator's top-10 univariate features — how far do the strongest
    single features already go?

This is a screening notebook: deltas here are indicative, not adoption
decisions. Anything that looks promising still has to win under the clean
baseline protocol (train_tail stop, seed 0) before it enters the pipeline.

In [1]:
import gc

import numpy as np
import pandas as pd
import xgboost as xgb
import yaml
from sklearn.metrics import roc_auc_score

HOURS = 168

In [2]:
DATA = "../data/kernel"
EXT = "../data/extended"

price = pd.read_csv(f"{DATA}/Historical_day_ahead_price_2015_2025.csv")
inflow = pd.read_csv(f"{DATA}/Historical_inflow_1958_2025.csv")
volume = pd.read_csv(f"{DATA}/Historical_volume_2015_2024.csv")
water = pd.read_csv(f"{DATA}/Synthetic_water_value_2015_2024.csv")
target = pd.read_csv(f"{DATA}/Unit_commitment_decisions.csv")
min_flow = pd.read_csv(f"{EXT}/Constraint_min_flow.csv")
min_volume = pd.read_csv(f"{EXT}/Constraint_min_volume.csv")
topology = yaml.safe_load(open(f"{EXT}/Tokke_Vinje_topology.yaml"))


def indexed(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"], utc=True, format="mixed")
    return df.sort_values("date").set_index("date")


price_h = indexed(price)["Day-ahead price (EUR/MWh)"].resample("1h").mean()
inf_wide = indexed(inflow)
vol_wide = indexed(volume)
wv_wide = indexed(water)
mf_wide = indexed(min_flow)
mv_wide = indexed(min_volume)

RESERVOIRS = list(vol_wide.columns)
CAPACITY = {r: float(topology["model"]["reservoir"][r]["max_vol"]) for r in RESERVOIRS}

target["starttime"] = pd.to_datetime(target["starttime"], utc=True, format="mixed")
target = target.sort_values("starttime").reset_index(drop=True)

GENERATORS = sorted(
    {c.removeprefix("result_committed_").rsplit("_t", 1)[0]
     for c in target.columns if c.startswith("result_committed")}
)

connections = topology["connections"]


def supplying_reservoirs(name, kind):
    if kind == "reservoir":
        return {name}
    out = set()
    for e in connections:
        if e["to"] == name and e["to_type"] == kind and e["from_type"] in {"reservoir", "tunnel"}:
            out |= supplying_reservoirs(e["from"], e["from_type"])
    return out


FEEDERS = {}
for gen in GENERATORS:
    plant = next(e["to"] for e in connections
                 if e["from"] == gen and e["from_type"] == "generator" and e["to_type"] == "plant")
    FEEDERS[gen] = sorted(supplying_reservoirs(plant, "plant"))

print("cases:", len(target), " generators:", len(GENERATORS))

cases: 2922  generators: 14


## Feature engineering and split

What is tested: the unchanged 193-column feature set and the baseline
split (train 2015-2021, val 2022), copied verbatim from
`notebooks/xgboost_baseline.ipynb`.

In [3]:
def lookup(frame, timestamps, columns=None):
    f = frame if columns is None else frame[columns]
    return f.reindex(pd.DatetimeIndex(timestamps)).to_numpy(dtype=np.float32)


def feature_matrix(cases):
    starts = pd.DatetimeIndex(cases["starttime"])
    n = len(starts)
    ts = pd.DatetimeIndex(np.repeat(starts.to_numpy(), HOURS)) + pd.to_timedelta(
        np.tile(np.arange(HOURS), n), unit="h"
    )
    price = lookup(price_h, ts).reshape(n, HOURS)
    initial = lookup(vol_wide, starts, RESERVOIRS)
    terminal = lookup(wv_wide, starts + pd.Timedelta(hours=HOURS), RESERVOIRS)
    inflow = lookup(inf_wide, ts).reshape(n, HOURS, -1)
    minflow = lookup(mf_wide, ts).reshape(n, HOURS, -1)
    minvol = lookup(mv_wide, ts).reshape(n, HOURS, -1)

    F = {}

    def add(name, v):
        v = np.asarray(v, dtype=np.float32)
        if v.shape == (n,):
            v = np.repeat(v, HOURS)
        elif v.shape == (n, HOURS):
            v = v.reshape(-1)
        F[name] = v

    h = np.tile(np.arange(HOURS, dtype=np.float32), n)
    add("horizon_hour", h)
    add("hours_remaining", HOURS - 1 - h)
    add("hour_of_day", ts.hour.to_numpy())
    add("day_of_week", ts.dayofweek.to_numpy())
    add("month", ts.month.to_numpy())
    add("doy_sin", np.sin(2 * np.pi * ts.dayofyear.to_numpy() / 365.25))
    add("doy_cos", np.cos(2 * np.pi * ts.dayofyear.to_numpy() / 365.25))

    add("price", price)
    for stat, v in (("mean", price.mean(1)), ("std", price.std(1)),
                    ("min", price.min(1)), ("max", price.max(1))):
        add(f"price_week_{stat}", v)
    add("price_minus_week_mean", price - price.mean(1, keepdims=True))
    add("price_week_rank", pd.DataFrame(price).rank(axis=1, pct=True).to_numpy())
    daily = price.reshape(n, 7, 24)
    for stat, v in (("mean", daily.mean(2)), ("min", daily.min(2)), ("max", daily.max(2))):
        add(f"price_day_{stat}", np.repeat(v, 24, axis=1))
    add("price_prefix_mean", price.cumsum(1) / np.arange(1, HOURS + 1))
    add("price_suffix_mean", price[:, ::-1].cumsum(1)[:, ::-1] / np.arange(HOURS, 0, -1))
    for off in (-24, -6, -1, 1, 6, 24):
        add(f"price_offset_{off:+d}", price[:, np.clip(np.arange(HOURS) + off, 0, HOURS - 1)])

    for j, res in enumerate(RESERVOIRS):
        add(f"init_vol__{res}", initial[:, j])
        add(f"init_frac__{res}", initial[:, j] / CAPACITY[res])
        add(f"term_val__{res}", terminal[:, j])
        add(f"price_minus_val__{res}", price - terminal[:, j, None])

    for j, name in enumerate(inf_wide.columns):
        flow = inflow[:, :, j]
        cum = flow.cumsum(1) * (3600 / 1e6)
        add(f"inflow__{name}", flow)
        add(f"inflow_week_mean__{name}", flow.mean(1))
        add(f"inflow_cum_Mm3__{name}", cum)
        add(f"inflow_rem_Mm3__{name}", cum[:, -1:] - cum)

    for j, name in enumerate(mf_wide.columns):
        add(f"minflow__{name}", minflow[:, :, j])
        add(f"minflow_week_max__{name}", minflow[:, :, j].max(1))
    for j, name in enumerate(mv_wide.columns):
        add(f"minvol__{name}", minvol[:, :, j])
        add(f"minvol_week_max__{name}", minvol[:, :, j].max(1))
        add(f"init_margin_to_minvol__{name}", initial[:, RESERVOIRS.index(name), None] - minvol[:, :, j])

    return pd.DataFrame(F, dtype=np.float32)


def generator_features(base, gen):
    X = base.copy()
    feeders = FEEDERS[gen]
    capacity = sum(CAPACITY[r] for r in feeders)
    total_init = sum(base[f"init_vol__{r}"] for r in feeders)
    value = sum(base[f"term_val__{r}"] * CAPACITY[r] for r in feeders) / capacity
    cum = sum(base[f"inflow_cum_Mm3__{r}"] for r in feeders)
    rem = sum(base[f"inflow_rem_Mm3__{r}"] for r in feeders)

    X["local_init_frac"] = total_init / capacity
    X["local_term_val"] = value
    X["local_price_minus_val"] = base["price"] - value
    X["local_inflow"] = sum(base[f"inflow__{r}"] for r in feeders)
    X["local_cum_frac"] = cum / capacity
    X["local_rem_frac"] = rem / capacity
    X["local_available_frac"] = (total_init + cum + rem) / capacity
    return X


def labels_for(cases, gen):
    cols = [f"result_committed_{gen}_t{t}" for t in range(HOURS)]
    return cases[cols].to_numpy(dtype=np.uint8).reshape(-1)


split = pd.Timestamp("2022-01-01", tz="UTC")
train_cases = target[target["starttime"] < split].reset_index(drop=True)
val_cases = target[target["starttime"] >= split].reset_index(drop=True)

Xtr_base = feature_matrix(train_cases)
Xva_base = feature_matrix(val_cases)
print("train matrix:", Xtr_base.shape, " val matrix:", Xva_base.shape)

train matrix: (429576, 186)  val matrix: (61320, 186)


## 1. Univariate signal: each feature alone

What is tested: for every feature and every generator, the val-2022 AUC of
using the raw feature value as the score, direction-corrected with
max(auc, 1-auc). Printed: the top 20 features by mean AUC across
generators, and each generator's top 10 features.

In [4]:
FEATURES = list(Xva_base.columns)
uni = pd.DataFrame(index=FEATURES, columns=GENERATORS, dtype=np.float32)
for gen in GENERATORS:
    yva = labels_for(val_cases, gen)
    if np.unique(yva).size < 2:
        continue
    for col in FEATURES:
        auc = roc_auc_score(yva, Xva_base[col].to_numpy())
        uni.loc[col, gen] = max(auc, 1.0 - auc)

uni["mean"] = uni.mean(axis=1)
print("top 20 features by mean univariate AUC across generators:")
print(uni["mean"].sort_values(ascending=False).head(20).round(4).to_string())

rows = []
for gen in GENERATORS:
    top10 = uni[gen].astype(float).sort_values(ascending=False).head(10).index.tolist()
    rows.append({"generator": gen, **{f"top{k + 1}": name for k, name in enumerate(top10)}})
pd.DataFrame(rows).set_index("generator")

top 20 features by mean univariate AUC across generators:
price_minus_val__Vinjevatn        0.8258
price_minus_val__Songavatn        0.8077
price_minus_val__Bitdalsvatn      0.8077
price_minus_val__Staavatn         0.7865
price_minus_val__Kjelavatn        0.7865
price_minus_val__Byrtevatn        0.7826
price_minus_val__Venemo           0.7739
price_minus_val__Hyljelihyl       0.7739
price_minus_val__Vaamarvatn       0.7739
price_minus_val__Totak            0.7739
term_val__Vinjevatn               0.7710
price_minus_val__Vatjern          0.7447
price_minus_val__Langeidvatn      0.7447
price_minus_val__Bandak           0.7407
term_val__Bandak                  0.7372
price_minus_val__Langesae         0.7241
price_minus_val__Foersvatn        0.7241
price_minus_val__Bordalsvatn      0.7241
term_val__Byrtevatn               0.7220
price_minus_val__Botnedalsvatn    0.7148


,top1,top2,top3,top4,top5,top6,top7,top8,top9,top10
generator,,,,,,,,,,
Byrte_G1,price_minus_val__Botnedalsvatn,term_val__Botnedalsvatn,price_minus_val__Totak,price_minus_val__Hyljelihyl,price_minus_val__Venemo,price_minus_val__Vaamarvatn,term_val__Foersvatn,term_val__Langesae,term_val__Bordalsvatn,price_minus_val__Bordalsvatn
Haukeli_G1,price_minus_val__Vinjevatn,term_val__Vinjevatn,price_minus_val__Byrtevatn,price_minus_val__Bitdalsvatn,price_minus_val__Songavatn,inflow_week_mean__Vatjern,inflow_week_mean__Foersvatn,inflow_week_mean__Hyljelihyl,inflow_week_mean__r_Vest_Vassdraget,inflow_week_mean__Bordalsvatn
Hogga_G1,price_minus_val__Bandak,term_val__Bandak,price_minus_val__Byrtevatn,price_minus_val__Bitdalsvatn,price_minus_val__Songavatn,price_minus_val__Vinjevatn,price_minus_val__Kjelavatn,price_minus_val__Staavatn,price_minus_val__Bordalsvatn,price_minus_val__Langesae
Kjela_G1,price_minus_val__Bordalsvatn,price_minus_val__Foersvatn,price_minus_val__Langesae,term_val__Foersvatn,term_val__Langesae,term_val__Bordalsvatn,price_minus_val__Venemo,price_minus_val__Vaamarvatn,price_minus_val__Totak,price_minus_val__Hyljelihyl
Lio_G1,price_minus_val__Byrtevatn,price_minus_val__Songavatn,price_minus_val__Bitdalsvatn,inflow_week_mean__Bandak,price_minus_val__Langesae,price_minus_val__Bordalsvatn,price_minus_val__Foersvatn,init_frac__Langesae,init_vol__Langesae,price_minus_val__Bandak
Songa_G1,price_minus_val__Bitdalsvatn,price_minus_val__Songavatn,price_minus_val__Venemo,price_minus_val__Hyljelihyl,price_minus_val__Vaamarvatn,price_minus_val__Totak,price_minus_val__Staavatn,price_minus_val__Kjelavatn,price_minus_val__Byrtevatn,price_minus_val__Bordalsvatn
Tokke_G1,price_minus_val__Vinjevatn,term_val__Vinjevatn,price_minus_val__Bitdalsvatn,price_minus_val__Songavatn,price_minus_val__Byrtevatn,price_minus_val__Staavatn,price_minus_val__Kjelavatn,term_val__Byrtevatn,price_minus_val__Totak,price_minus_val__Vaamarvatn
Tokke_G2,price_minus_val__Vinjevatn,term_val__Vinjevatn,price_minus_val__Songavatn,price_minus_val__Bitdalsvatn,price_minus_val__Byrtevatn,price_minus_val__Kjelavatn,price_minus_val__Staavatn,price_minus_val__Venemo,price_minus_val__Totak,price_minus_val__Vaamarvatn
Tokke_G3,price_minus_val__Vinjevatn,term_val__Vinjevatn,price_minus_val__Bitdalsvatn,price_minus_val__Songavatn,price_minus_val__Byrtevatn,price_minus_val__Staavatn,price_minus_val__Kjelavatn,price_minus_val__Hyljelihyl,price_minus_val__Venemo,price_minus_val__Vaamarvatn


## 2. Feature-group ablation with a default XGBoost

What is tested: per-generator `xgb.XGBClassifier(tree_method='hist')` with
all-default values (100 trees, depth 6, lr 0.3, no early stopping), fitted
on train 2015-2021 and scored on val 2022 — once with all 193 features and
once per feature group dropped. Printed: micro AUC per configuration and
the drop vs the all-features run. The size of the drop measures how much
signal the group carries on top of the rest.

In [ ]:
def group_of(col):
    if col.startswith("local_"):
        return "local"
    if col.startswith(("minflow", "minvol", "init_margin_to_minvol")):
        return "constraints"
    if col.startswith(("init_vol__", "init_frac__", "term_val__", "price_minus_val__")):
        return "reservoir"
    if col.startswith("inflow"):
        return "inflow"
    if col.startswith("price"):
        return "price"
    return "calendar"


GROUPS = ["calendar", "price", "reservoir", "inflow", "constraints", "local"]


def micro_default(drop=None, columns_fn=None):
    preds = {}
    for gen in GENERATORS:
        Xg = generator_features(Xtr_base, gen)
        Xv = generator_features(Xva_base, gen)
        cols = [c for c in Xg.columns if group_of(c) != drop] if drop else None
        if columns_fn is not None:
            cols = columns_fn(gen, list(Xg.columns))
        ytr = labels_for(train_cases, gen)
        yva = labels_for(val_cases, gen)
        if np.unique(ytr).size == 1:
            preds[gen] = np.full(len(yva), float(ytr[0]), dtype=np.float32)
        else:
            clf = xgb.XGBClassifier(tree_method="hist", random_state=0)
            clf.fit(Xg[cols] if cols else Xg, ytr)
            preds[gen] = clf.predict_proba(Xv[cols] if cols else Xv)[:, 1].astype(np.float32)
            del clf
        del Xg, Xv
        gc.collect()
    y_all = np.concatenate([labels_for(val_cases, g) for g in GENERATORS])
    p_all = np.concatenate([preds[g] for g in GENERATORS])
    return roc_auc_score(y_all, p_all)


abl = {"all": micro_default()}
print(f"{'all':14s} micro AUC {abl['all']:.4f}")
for g in GROUPS:
    abl[f"-{g}"] = micro_default(drop=g)
    print(f"{'-' + g:14s} micro AUC {abl[f'-{g}']:.4f}  "
          f"(drop {abl['all'] - abl[f'-{g}']:+.4f})")

## 3. Top-10 univariate features only

What is tested: the same default XGBoost per generator, using only that
generator's top-10 features from step 1. Printed: micro AUC and the
per-generator AUC next to the all-features default run — how much of the
signal the strongest single features already carry.

In [ ]:
def top10_columns(gen):
    return uni[gen].astype(float).sort_values(ascending=False).head(10).index.tolist()


top10_micro = micro_default(columns_fn=lambda gen, all_cols: top10_columns(gen))
print(f"top-10 only: micro AUC {top10_micro:.4f}  "
      f"(all features: {abl['all']:.4f})")

rows = []
for gen in GENERATORS:
    yva = labels_for(val_cases, gen)
    cols = top10_columns(gen)
    Xg = generator_features(Xtr_base, gen)[cols]
    Xv = generator_features(Xva_base, gen)[cols]
    ytr = labels_for(train_cases, gen)
    if np.unique(ytr).size == 1:
        continue
    clf = xgb.XGBClassifier(tree_method="hist", random_state=0)
    clf.fit(Xg, ytr)
    p = clf.predict_proba(Xv)[:, 1]
    rows.append({"generator": gen, "top10_auc": roc_auc_score(yva, p)})
    del clf, Xg, Xv
pd.DataFrame(rows).set_index("generator").round(4)

NameError: name 'micro_default' is not defined